In [17]:
from langchain_ollama.llms import OllamaLLM
import os
import requests
import time
# from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import TextLoader

#model= OllamaLLM(model="llama3.2:3b")
model= OllamaLLM(model="qwen2.5:7b")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [3]:
loader = TextLoader("./HP1RUS.txt")
documents = loader.load()

In [5]:
# documents[]

In [4]:
# ============================================================
# ШАГ 5: РАЗБИЕНИЕ НА ЧАНКИ
# ============================================================
# print("\n🧩 Разбиение на чанки...")
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,
#     chunk_overlap=200,
# )
# splits = text_splitter.split_documents(documents)
# print(f"✅ Получено {len(splits)} чанков")

# 2. Разбиваем на чанки
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,          # Размер одного чанка в символах
    chunk_overlap=200,        # Перекрытие между чанками (чтобы не терять контекст на границах)
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # Разбиваем по абзацам, потом по предложениям
)

chunks = text_splitter.split_documents(documents)
print(f"Получено {len(chunks)} чанков")


Получено 554 чанков


In [5]:
# ============================================================
# ШАГ 6: СОЗДАНИЕ ИНДЕКСА (Chroma) — БЕЗ persist()!
# ============================================================
print("\n💾 Создание индекса Chroma...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db_rag"  # Индекс автоматически сохранится сюда
)
# ❌ НЕ НУЖНО: vectorstore.persist()
print("✅ Индекс создан и сохранен в папку ./chroma_db_rag")


💾 Создание индекса Chroma...
✅ Индекс создан и сохранен в папку ./chroma_db_rag


In [54]:
from langchain_core.output_parsers import StrOutputParser
# ============================================================
# ШАГ 7: СОЗДАНИЕ РЕТРИВЕРА
# ============================================================
retriever = vectorstore.as_retriever(search_kwargs={"k": 7})
print("✅ Ретривер готов (поиск 3 чанков)")

# ============================================================
# ШАГ 8: RAG ЦЕПОЧКА
# ============================================================
print("\n🔗 Сборка RAG цепочки...")
# PROMPT_TEXT="""
# ВАЖНО: Отвечай ТОЛЬКО на русском языке. Никаких других языков.
# Ты — эксперт по книгам о Гарри Поттере. Отвечай на русском языке.

# Инструкции:
# 1. Используй ТОЛЬКО информацию из контекста.
# 2. Если в контексте есть шутка, ирония или намёк — воспринимай это как ответ.
# 3. Отвечай кратко и по делу.
# 4. Если информации совсем нет — скажи "В контексте нет ответа".

# Контекст: {context}

# Вопрос: {question}
# """
PPROMPT_TEXT="""
Ты — эксперт по книгам о Гарри Поттере.
ОТВЕЧАЙ ТОЛЬКО НА РУССКОМ ЯЗЫКЕ.
Используй ТОЛЬКО информацию из контекста.
ЕСЛИ В КОНТЕКСТЕ НЕТ ОТВЕТА — скажи "В контексте нет ответа".
НЕ ПРИДУМЫВАЙ информацию, которой нет в контексте.
Контекст: {context}

Вопрос: {question}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", PROMPT_TEXT)
])

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

print("✅ RAG цепочка готова!")

✅ Ретривер готов (поиск 3 чанков)

🔗 Сборка RAG цепочки...
✅ RAG цепочка готова!


In [56]:
#question = "Кто сжег все письма"
#question = "Кто такие Дарсли?"
# question="Почему Хагрид не превратил Дадли в свинью?"
question="Кто отвез Гарри в Хогвартс?"
#question="На какой платформе Гарри должен был сесть на поезд?"

print(question )
answer = rag_chain.invoke(question)
print(answer)


Кто отвез Гарри в Хогвартс?
В контексте нет ответа.


In [57]:
QUESTIONS = [
    {"id": 1, "question": "Как зовут сову Гарри?", "expected": "Хедвиг"},
    {"id": 2, "question": "Кто отвез Гарри в Хогвартс?", "expected": "Дядя Вернон отвёз на вокзал, поезд"},
    {"id": 3, "question": "Как магглы называют не магических людей?", "expected": "Магглы"},
    {"id": 4, "question": "На какой платформе Гарри должен был сесть на поезд?", "expected": "9 и 3/4"},
    {"id": 5, "question": "Кто написал записку Дамблдору?", "expected": "Хагрид"},
    {"id": 6, "question": "Почему Хагрид не превратил Дадли в свинью?", "expected": "похож на свинью"},
    {"id": 7, "question": "Боится ли Дадли Гарри после случившегося?", "expected": "да, не оставался в одной комнате"},
    {"id": 8, "question": "Почему Гарри сидел на подоконнике и думал?", "expected": "размышлял о жизни"},
    {"id": 9, "question": "Как дядя Вернон отреагировал на Хагрида?", "expected": "бледный, злой, Он не поедет"},
    {"id": 10, "question": "Почему Невилл наколдовал искры?", "expected": "Малфой схватил за шиворот"},
    {"id": 11, "question": "Как изменилось отношение Дарсли к Гарри после визита Хагрида?", "expected": "игнорируют, не запирают"},
    {"id": 12, "question": "Какие магические способности проявили дети в лесу?", "expected": "Невилл наколдовал искры"},
    {"id": 13, "question": "Почему Хагрид был исключен из Хогвартса?", "expected": "сломал волшебную палочку"},
    {"id": 14, "question": "Какая погода была, когда Хагрид отправил письмо?", "expected": "шторм"},
    {"id": 15, "question": "Что чувствовал Гарри, когда поезд тронулся?", "expected": "хотел смотреть на Хагрида"},
    {"id": 16, "question": "Хагрид убил Волан-де-Морта?", "expected": "нет"},
    {"id": 17, "question": "Дадли стал волшебником?", "expected": "нет, маггл"},
    {"id": 18, "question": "Гарри получил письмо из Хогвартса до визита Хагрида?", "expected": "да, Дарсли скрывали"},
    {"id": 19, "question": "Хагрид использовал палочку в доме Дарсли?", "expected": "да, зонт"},
    {"id": 20, "question": "Клык - это волшебное существо?", "expected": "нет, пёс"},
    {"id": 21, "question": "Сравни поведение Хагрида и дяди Вернона", "expected": "Хагрид спокоен, Вернон злой"},
    {"id": 22, "question": "Что общего между Невиллом и Гарри?", "expected": "учатся в Хогвартсе"},
    {"id": 23, "question": "Как Хагрид отправил письмо Дамблдору?", "expected": "через сову"},
    {"id": 24, "question": "Почему дядя Вернон был бледным?", "expected": "страх, гнев"},
    {"id": 25, "question": "Куда Хагрид обещал отвезти Гарри завтра?", "expected": "купить всё к школе"},
]
for item in QUESTIONS:
    q=item.get("question")
    answer = rag_chain.invoke(q)
    print("Вопрос",q )
    print("Ответ",answer)

Вопрос Как зовут сову Гарри?
Ответ В контексте нет ответа.
Вопрос Кто отвез Гарри в Хогвартс?
Ответ В контексте нет ответа.
Вопрос Как магглы называют не магических людей?
Ответ В контексте нет ответа.
Вопрос На какой платформе Гарри должен был сесть на поезд?
Ответ Гарри должен был сесть на платформу девять и три четверти.
Вопрос Кто написал записку Дамблдору?
Ответ В контексте нет ответа.
Вопрос Почему Хагрид не превратил Дадли в свинью?
Ответ Хагрид не смог превратить Дадли в свинью потому, что тот был слишком похож на свинтуса сам. В контексте нет других подробностей об этой ситуации.
Вопрос Боится ли Дадли Гарри после случившегося?
Ответ Нет, Дадли все еще продолжает бить Гарри и не боится его. В тексте говорится, что Гарри очень быстрый, но обычно Дадли не может его поймать из-за этого.
Вопрос Почему Гарри сидел на подоконнике и думал?
Ответ В контексте нет ответа.
Вопрос Как дядя Вернон отреагировал на Хагрида?
Ответ Дядя Вернон зарычал и забрал тетю Петунью и Дадли в соседнюю к

In [50]:
results = retriever.invoke(question)

# Смотришь результаты
for doc in results:
    print(doc.page_content)
    print("---")

каждый хруст веточки. Что происходит? Где остальные? 
   Наконец, громкий треск возвестил возвращение Хагрида. За ним шли Малфой, 
Невилл и Клык. Хагрид кипел. Оказалось, Малфой подкрался к Невиллу и в шутку 
схватил его за шиворот. Невилл запаниковал и наколдовал искры. 
   "Нам очень повезет, если мы его поймаем с тем балаганом, который вы тут 
устроили. Так, поменяемся - Невилл, ты остаешься со мной и Эрмионой, Гарри ты 
берешь Клыка и этого идиота. Извини, - шепотом добавил Хагрид, - будет потрудней 
напугать тебя, а нам нужно сделать дело". 
   Таким образом Гарри отправился в чащу леса с Малфоем и Клыком. Они шли почти 
полчаса, дальше и дальше, до тех пор, пока по тропинке уже нельзя было пройти - 
так густо здесь росли деревья. Гарри казалось, что крови прибавилось. Капельки 
виднелись на корнях дерева, как будто бедное создание устало прислонялось к 
нему. Через спутанные ветви старого дуба, Гарри увидел впереди поляну.
---
Когда Гарри и Рон возвращались в замок к ужину, с кар

In [25]:
# # results = retriever.invoke(question)
# embeddings_search = OllamaEmbeddings(model="nomic-embed-text")  # Та же!
# retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
# #retriever = vectorstore.as_retriever()
# docs = retriever.get_relevant_documents(question)

# # Смотришь результаты
# for doc in results:
#     print(doc.page_content)
#     print("---")

AttributeError: 'VectorStoreRetriever' object has no attribute 'get_relevant_documents'